# 02 — Feature Engineering
## Actuarial-Driven Feature Creation

This notebook demonstrates the `ActuarialFeatureBuilder` pipeline:
- Premium features (pricing adequacy)
- Claims experience features
- Loyalty & tenure features
- Portfolio (multi-line) features
- Lifecycle & channel features

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.features.actuarial_features import ActuarialFeatureBuilder, FEATURE_DESCRIPTIONS

sns.set_theme(style="whitegrid")
DATA_PATH = Path("../data/churn_dataset.parquet")

In [ ]:
df = pd.read_parquet(DATA_PATH)
y = df["churn_label"]
X = df.drop(columns=["churn_label"])
print(f"Raw features: {X.shape[1]}")
X.head()

## Apply ActuarialFeatureBuilder

In [ ]:
builder = ActuarialFeatureBuilder()
X_features = builder.fit_transform(X)

new_cols = [c for c in X_features.columns if c not in X.columns]
print(f"Engineered features: {len(new_cols)}")
print("\nNew features:")
for col in new_cols:
    desc = FEATURE_DESCRIPTIONS.get(col, "")
    print(f"  {col:30s} — {desc}")

## Feature Distributions (Engineered)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i, col in enumerate(new_cols[:12]):
    ax = axes[i // 4, i % 4]
    for label, color in [(0, "steelblue"), (1, "coral")]:
        subset = X_features[y == label][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f"{'Retained' if label == 0 else 'Churned'}")
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle("Engineered Feature Distributions by Churn Status", fontsize=14)
plt.tight_layout()
plt.show()

## Feature Importance Preview (Correlation with Target)

In [ ]:
numeric_features = X_features[new_cols].select_dtypes(include=[np.number])
corr_with_target = numeric_features.corrwith(y).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["coral" if v > 0 else "steelblue" for v in corr_with_target.values]
corr_with_target.plot.barh(ax=ax, color=colors)
ax.set_xlabel("Pearson Correlation with Churn")
ax.set_title("Feature Correlation with Churn Label")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## Market Rate Premiums Learned by fit()

In [ ]:
print("Market average premiums per LOB (learned from training data):")
for lob, premium in sorted(builder.market_avg_premiums_.items()):
    print(f"  {lob:12s}: €{premium:,.0f}")

## Summary

The `ActuarialFeatureBuilder` adds 15+ domain-specific features that capture:
- **Pricing adequacy** — the strongest churn driver
- **Claims satisfaction** — negative experience pushes churn
- **Loyalty signals** — tenure and renewal history
- **Portfolio bundling** — multi-line customers are sticky
- **Lifecycle stage** — age and channel effects